In [1]:
#Install requirements
%pip install "transformers==4.44.2" "tokenizers==0.19.1" -q
%pip install "FlagEmbedding==1.3.5" -q

%pip install pymupdf beautifulsoup4 requests mitreattack-python \
            langchain-text-splitters numpy pandas tqdm chromadb \
            "ollama>=0.3" "openai>=1.30" \
            ipykernel jupyter -q

%pip install git+https://github.com/vibrantlabsai/ragas -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
import pickle
import numpy as np
import pandas as pd
from collections import Counter
import asyncio

import pymupdf
import requests
from bs4 import BeautifulSoup
from mitreattack.stix20 import MitreAttackData
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from FlagEmbedding import BGEM3FlagModel, FlagReranker

import ollama

from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import Faithfulness, ContextUtilization, AnswerRelevancy

from tqdm.notebook import tqdm

In [3]:
#Pull LLM and embedding models from ollama
!ollama pull qwen3:4b
!ollama pull nomic-embed-text

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 3e4cb1417446: 100% ▕██████████████████▏ 2.5 GB                         
pulling 2d54db2b9bb2: 100% ▕██████████████████▏ 1.5 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling cff3f395ef37: 100% ▕██████████████████▏  120 B                         
pulling e18a783aae55: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ p

In [4]:
#Load MITRE ATT&CK data
attacks = MitreAttackData("docs/enterprise_attack.json")

# Build lookup table
technique_to_mitigations = attacks.get_all_mitigations_mitigating_all_techniques()
technique_to_software = attacks.get_all_software_using_all_techniques()

techniques = attacks.get_techniques(
    include_subtechniques=True,
    remove_revoked_deprecated=True,
)

print(f"Total techniques: {len(techniques)}")

Total techniques: 691


In [5]:
# build ATT&CK chunks
attck_chunks = []

#extract metadata from techniques and add as chunks
for technique in techniques:
    mitre_id = attacks.get_attack_id(technique.id) or ""
    tactics = [t.name for t in attacks.get_tactics_by_technique(technique.id)]

    software_entries = technique_to_software.get(technique.id, [])
    mitigation_entries = technique_to_mitigations.get(technique.id, [])

    tools = [e["object"].name for e in software_entries if e["object"].type == "tool"]
    malware = [
        e["object"].name for e in software_entries if e["object"].type == "malware"
    ]
    mitigation_names = [e["object"].name for e in mitigation_entries]
    is_subtechnique = "." in mitre_id

    chunk_text = f"""MITRE ATT&CK Technique: {technique.name}
ID: {mitre_id}
Type: {"Sub-technique" if is_subtechnique else "Technique"}
Tactics: {", ".join(tactics)}

Description:
{getattr(technique, "description", "Not available").strip()}

Detection:
{getattr(technique, "x_mitre_detection", "Not documented").strip()}

Associated Tools: {", ".join(tools) if tools else "None documented"}
Associated Malware: {", ".join(malware) if malware else "None documented"}
Mitigations: {", ".join(mitigation_names) if mitigation_names else "None documented"}""".strip()

    attck_chunks.append(
        {
            "text": chunk_text,
            "metadata": {
                "source": "MITRE ATT&CK",
                "id": mitre_id,
                "name": technique.name,
                "tactics": tactics,
                "is_subtechnique": is_subtechnique,
                "tools": tools,
                "malware": malware,
                "mitigations": mitigation_names,
            },
        }
    )

print(f"ATT&CK chunks built: {len(attck_chunks)}")
sample = attck_chunks[0]
print(f"\nSample — {sample['metadata']['id']}: {sample['metadata']['name']}")
print(sample["text"][:400])

ATT&CK chunks built: 691

Sample — T1055.011: Extra Window Memory Injection
MITRE ATT&CK Technique: Extra Window Memory Injection
ID: T1055.011
Type: Sub-technique
Tactics: Privilege Escalation, Defense Evasion

Description:
Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate 


In [ ]:
PDF_SOURCES = {
    "docs/cisa_AA24-326A.pdf": "CISA Red Team Assessment AA24-326A",
    "docs/cisa_aa24-193a_silentshield.pdf": "CISA Red Team Assessment AA24-193A SILENTSHIELD",
    "docs/cisa_aa23-059a.pdf": "CISA Red Team Assessment AA23-059A",
    "docs/nist_sp_800-115.pdf": "NIST SP 800-115 Information Security Testing",
    "docs/nist_sp_800-61r3.pdf": "NIST SP 800-61r3 Incident Response",
    "docs/nist_sp_800-53r5.pdf": "NIST SP 800-53r5 Security and Privacy Controls",
    "docs/nist_csf_2.pdf": "NIST Cybersecurity Framework 2.0",
    "docs/cert_vuln_disclosure.pdf": "CERT Guide to Coordinated Vulnerability Disclosure",
    "docs/tiber-eu.pdf": "EU Threat Intelligence-Based Ethical Red Teaming",
    "docs/cbest_tila.pdf": "CBEST Threat Intelligence-Led Assessments",
}


#Extract text from PDFs and split into overlapping chunks.
def parse_and_chunk_pdf(path, source_name, chunk_size=800, overlap=150):
    #Read pdf, split in order of separators. Chunk to 800 chars with an overlap of 150.
    doc = pymupdf.open(path)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " "],
    )
    chunks = []
    #Extract content from each page in a document.
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if not text:
            continue
        #chunk text based on splitter criteria
        for j, split in enumerate(splitter.split_text(text)):
            #remove chunks less than 50 chars
            if len(split.strip()) < 50:
                continue
            chunks.append(
                {
                    "text": split.strip(),
                    "metadata": {
                        "source": source_name,
                        "page": i + 1,
                        "chunk_index": j,
                    },
                }
            )
    return chunks


pdf_chunks = []
for path, source_name in PDF_SOURCES.items():
    chunks = parse_and_chunk_pdf(path, source_name)
    pdf_chunks.extend(chunks)
    print(f"{source_name}: {len(chunks)} chunks")

print(f"\nTotal PDF chunks: {len(pdf_chunks)}")

CISA Red Team Assessment AA24-326A: 135 chunks
CISA Red Team Assessment AA24-193A SILENTSHIELD: 109 chunks
CISA Red Team Assessment AA23-059A: 99 chunks
MuPDF error: format error: No default Layer config

NIST SP 800-115 Information Security Testing: 419 chunks
NIST SP 800-61r3 Incident Response: 177 chunks
NIST SP 800-53r5 Security and Privacy Controls: 2649 chunks
NIST Cybersecurity Framework 2.0: 115 chunks
CERT Guide to Coordinated Vulnerability Disclosure: 535 chunks
EU Threat Intelligence-Based Ethical Red Teaming: 233 chunks
CBEST Threat Intelligence-Led Assessments: 149 chunks

Total PDF chunks: 4620


In [7]:
PTES_SECTIONS = {
    "Pre-engagement Interactions": "http://www.pentest-standard.org/index.php/Pre-engagement",
    "Intelligence Gathering": "http://www.pentest-standard.org/index.php/Intelligence_Gathering",
    "Threat Modeling": "http://www.pentest-standard.org/index.php/Threat_Modeling",
    "Vulnerability Analysis": "http://www.pentest-standard.org/index.php/Vulnerability_Analysis",
    "Exploitation": "http://www.pentest-standard.org/index.php/Exploitation",
    "Post Exploitation": "http://www.pentest-standard.org/index.php/Post_Exploitation",
    "Reporting": "http://www.pentest-standard.org/index.php/Reporting",
}

#Web pages are scraped, ensuring a user agent is specified with a timeout to prevent being rate limited.
def scrape_ptes_section(url, section_name):
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        r.raise_for_status()
    except Exception as e:
        print(f"  Failed to fetch {section_name}: {e}")
        return None

    #extract div and body tag content.
    soup = BeautifulSoup(r.text, "html.parser")
    content_div = soup.find("div", {"id": "mw-content-text"}) or soup.find("body")
    if not content_div:
        return None

    #with the content, find the following tags
    for tag in content_div.find_all(
        ["script", "style", "table", "sup", "span.editsection"]
    ):
        tag.decompose()

    #extract lines and clean up text
    lines = [l.strip() for l in content_div.get_text(separator="\n").splitlines()]
    clean_text = "\n\n".join(l for l in lines if l)
    return clean_text


def chunk_ptes_section(text, section_name, chunk_size=800, overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " "],
    )
    return [
        {
            "text": s.strip(),
            "metadata": {"source": "PTES", "section": section_name, "chunk_index": i},
        }
        for i, s in enumerate(splitter.split_text(text))
        if len(s.strip()) >= 50
    ]


ptes_chunks = []
for section_name, url in PTES_SECTIONS.items():
    print(f"Scraping: {section_name}...")
    text = scrape_ptes_section(url, section_name)
    if text:
        chunks = chunk_ptes_section(text, section_name)
        ptes_chunks.extend(chunks)
        print(f"{len(chunks)} chunks")

print(f"\nTotal PTES chunks: {len(ptes_chunks)}")

Scraping: Pre-engagement Interactions...
72 chunks
Scraping: Intelligence Gathering...
75 chunks
Scraping: Threat Modeling...
35 chunks
Scraping: Vulnerability Analysis...
49 chunks
Scraping: Exploitation...
30 chunks
Scraping: Post Exploitation...
56 chunks
Scraping: Reporting...
22 chunks

Total PTES chunks: 339


In [ ]:
#combine chunks
all_chunks = attck_chunks + pdf_chunks + ptes_chunks

print("Chunks per source:")
for source, count in Counter(c["metadata"]["source"] for c in all_chunks).items():
    print(f"{source}: {count}")
print(f"TOTAL: {len(all_chunks)}")

Chunks per source:
  MITRE ATT&CK: 691
  CISA Red Team Assessment AA24-326A: 135
  CISA Red Team Assessment AA24-193A SILENTSHIELD: 109
  CISA Red Team Assessment AA23-059A: 99
  NIST SP 800-115 Information Security Testing: 419
  NIST SP 800-61r3 Incident Response: 177
  NIST SP 800-53r5 Security and Privacy Controls: 2649
  NIST Cybersecurity Framework 2.0: 115
  CERT Guide to Coordinated Vulnerability Disclosure: 535
  EU Threat Intelligence-Based Ethical Red Teaming: 233
  CBEST Threat Intelligence-Led Assessments: 149
  PTES: 339
  TOTAL: 5650


## Embedding

In [9]:
# Load BGE-M3 embedding model
embedder = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("BGE-M3 loaded")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

BGE-M3 loaded


In [10]:
#Ensure max chuck size is 3000 chars to reduce token usage.
def truncate_chunk(text, max_chars=3000):
    return text[:max_chars] if len(text) > max_chars else text

all_texts = [truncate_chunk(c["text"]) for c in all_chunks]
all_dense = []
all_sparse = []
BATCH_SIZE = 16

for start in tqdm(range(0, len(all_texts), BATCH_SIZE), desc="Embedding chunks"):
    batch = all_texts[start : start + BATCH_SIZE]

    #Encode chunks using the BGE-M3 embedder in batch sizes of 16. Return dense and sparse embeddings.
    encoded = embedder.encode(
        batch, return_dense=True, return_sparse=True, batch_size=BATCH_SIZE
    )

    all_dense.append(encoded["dense_vecs"])
    all_sparse.extend(encoded["lexical_weights"])

dense_vecs = np.vstack(all_dense)
sparse_weights = all_sparse

print(f"Dense shape:    {dense_vecs.shape}")
print(f"Sparse entries: {len(sparse_weights)}")

Embedding chunks:   0%|          | 0/354 [00:00<?, ?it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 109.68it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 21.51it/s]


Dense shape:    (5650, 1024)
Sparse entries: 5650


In [11]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")

#delete collection if it exists, otherwise create one with cosine similary as the distance function.
try:
    chroma_client.delete_collection("redteam_rag")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="redteam_rag", metadata={"hnsw:space": "cosine"}
)

ids = [str(i) for i in range(len(all_chunks))]
documents = [c["text"] for c in all_chunks]

#flatten metadata.
metadatas = []
for c in all_chunks:
    meta = dict(c["metadata"])
    for k, v in meta.items():
        if isinstance(v, list):
            meta[k] = ", ".join(v) if v else ""
    metadatas.append(meta)

#Pass batches into ChromaDB collection.s
BATCH_SIZE = 500
for i in range(0, len(all_chunks), BATCH_SIZE):
    end = min(i + BATCH_SIZE, len(all_chunks))
    collection.add(
        ids=ids[i:end],
        documents=documents[i:end],
        embeddings=dense_vecs[i:end].tolist(),
        metadatas=metadatas[i:end],
    )
    print(f"  Stored batch {i}:{end}")

print(f"Total stored in ChromaDB: {collection.count()}")

  Stored batch 0:500
  Stored batch 500:1000
  Stored batch 1000:1500
  Stored batch 1500:2000
  Stored batch 2000:2500
  Stored batch 2500:3000
  Stored batch 3000:3500
  Stored batch 3500:4000
  Stored batch 4000:4500
  Stored batch 4500:5000
  Stored batch 5000:5500
  Stored batch 5500:5650
Total stored in ChromaDB: 5650


In [12]:
#save sparse weights
sparse_store = {str(i): sparse_weights[i] for i in range(len(all_chunks))}

with open("sparse_weights.pkl", "wb") as f:
    pickle.dump(sparse_store, f)

print("Sparse weights saved")

Sparse weights saved


In [13]:
#Reload embedder, collection, and sparse weights
embedder = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection("redteam_rag")

with open("sparse_weights.pkl", "rb") as f:
    sparse_store = pickle.load(f)

print(f"Collection: {collection.count()} chunks")
print(f"Sparse:     {len(sparse_store)} entries")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Collection: 5650 chunks
Sparse:     5650 entries


## Retrieval

In [ ]:
#A hybrid retrieval (dense and sparse) will be used. The dense embedding outputs 
# the semantic similarity whereas the sparse embedding matches keywords.

#Top k chunks by cosine similarity are retrieved.
def dense_query(query_vec, collection, top_k=10):
    results = collection.query(
        query_embeddings=query_vec.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    ids = [int(i) for i in results["ids"][0]]
    scores = [1 - d for d in results["distances"][0]]
    return ids, scores

#Top k chunks based on shared tokens between query and chunk are retrieved.
def sparse_query(query_sparse, sparse_store, top_k=10):
    scores = {}
    for idx, doc_sparse in sparse_store.items():
        shared = set(query_sparse.keys()) & set(doc_sparse.keys())
        scores[int(idx)] = sum(query_sparse[t] * doc_sparse[t] for t in shared)
    sorted_ids = sorted(scores, key=scores.get, reverse=True)[:top_k]
    return sorted_ids, [scores[i] for i in sorted_ids]


#Reciprocal ranking ensures chunks with a lower rank are not totally ignored due to the 
# influence of the high scoring chunks. This is important as the top 10 chunks are 
# retrieved and while having lower scores than the top few, could be more relevant than 
# other chunks.
def reciprocal_rank_fusion(ranked_lists, k=60):
    rrf_scores = {}
    for ranked_ids in ranked_lists:
        for rank, doc_id in enumerate(ranked_ids):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return rrf_scores


def hybrid_retrieve(query, embedder, collection, sparse_store, all_chunks, top_k=10):
    #The query is encoded and it's dense and sparse embeddings are extracted.
    q_encoded = embedder.encode([query], return_dense=True, return_sparse=True)
    q_dense = q_encoded["dense_vecs"]
    q_sparse = q_encoded["lexical_weights"][0]

    #Extract top 10 chunks from dense and sparse searches
    dense_ids, _ = dense_query(q_dense, collection, top_k)
    sparse_ids, _ = sparse_query(q_sparse, sparse_store, top_k)

    #Retrieve scores from RRF and sort by score to get candidate top 10 chunks
    rrf_scores = reciprocal_rank_fusion([dense_ids, sparse_ids])
    sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]

    return [all_chunks[i] for i in sorted_ids], sorted_ids

In [ ]:
#Load reranker
reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)

#This function reranks the top 10 chunks obtained from the hybrid retriever and 
# returns the top 3 highest scoring chunks based on the BGE reranker.
def rerank(query, candidates, top_k=3):
    pairs = [[query, c["text"]] for c in candidates]
    scores = reranker.compute_score(pairs, normalize=True)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [c for _, c in ranked[:top_k]], [s for s, _ in ranked[:top_k]]

## Generation

In [16]:
#Prompt LLM with its role.
SYSTEM_PROMPT = """You are an expert red team cybersecurity advisor.
Your role is to provide accurate, actionable guidance to red team practitioners.
Answer ONLY based on the provided context.
If the context does not contain enough information, say so explicitly.
Always cite the source document (e.g. MITRE ATT&CK T1550.002, PTES, CISA)
for each piece of advice you give."""

#generate response using the retrieved context.
def generate(query, context_chunks, stream=True):
    context = "\n\n---\n\n".join(
        [
            f"[Source: {c['metadata']['source']} | "
            f"{c['metadata'].get('id', c['metadata'].get('section', ''))}]\n"
            f"{c['text']}"
            for c in context_chunks
        ]
    )

    #Setting temperature to 0.1 ensures the LLM is more fact-based.
    response_stream = ollama.chat(
        model="qwen3:4b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
        ],
        options={"temperature": 0.1, "think": False},
        stream=stream,
    )

    #Stream imitates typical LLM GUIs where responses are printed token-by-token.
    if stream:
        full_response = ""
        for chunk in response_stream:
            token = chunk["message"]["content"]
            print(token, end="", flush=True)
            full_response += token
        print()
        return full_response
    else:
        return response_stream["message"]["content"]

## RAG

Measures:
* [Faithfulness](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/faithfulness/)

$\text{Faithfulness Score} \in (0, 1) = \frac{\text{Number of claims in the response supported by the retrieved context}}{\text{Total number of claims in the response}}$

* [Context Utilisation](https://docs.ragas.io/en/v0.1.21/concepts/metrics/context_utilization.html)

$ x \in (0, 1)$

* [Answer Relevance](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/answer_relevance/)

$ x \in (-1, 1)$

In [17]:
def rag_pipeline(query, top_k_retrieve=10, top_k_rerank=3, verbose=True):
    if verbose:
        print(f"Query: {query}\n")

    #Run hybrid retrieval based on query. Dense and sparse searches return 10 chunks 
    # in total which are reranked and the top 10 chunks are returned
    candidates, _ = hybrid_retrieve(
        query, embedder, collection, sparse_store, all_chunks, top_k=top_k_retrieve
    )
    if verbose:
        print(f"Retrieved {len(candidates)} candidates")

    #Hybrid retrieved top 10 chunks are reranked and top 3 are outputted.
    final_chunks, rerank_scores = rerank(query, candidates, top_k=top_k_rerank)
    if verbose:
        print(f"Reranked to top {top_k_rerank}:")
        for i, (chunk, score) in enumerate(zip(final_chunks, rerank_scores)):
            src = chunk["metadata"]["source"]
            name = chunk["metadata"].get("id", chunk["metadata"].get("section", ""))
            print(f"  {i+1}. [{score:.3f}] {src} | {name}")

    #User query and top 3 chunks are passed into LLM and LLM response is returned.
    answer = generate(query, final_chunks, stream=verbose)
    if verbose:
        print(f"\nAnswer:\n{answer}")

    return {
        "query": query,
        "retrieved": candidates,
        "final_chunks": final_chunks,
        "rerank_scores": rerank_scores,
        "answer": answer,
    }


# Check if RAG pipeline is working.
_ = rag_pipeline(
    "What reconnaissance techniques should I use before an assumed breach engagement?"
)

Query: What reconnaissance techniques should I use before an assumed breach engagement?



You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Retrieved 10 candidates


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Reranked to top 3:
  1. [0.022] NIST SP 800-115 Information Security Testing | 
  2. [0.014] MITRE ATT&CK | T1597.002
  3. [0.014] MITRE ATT&CK | T1596.005
Based on the provided context, here are the appropriate reconnaissance techniques for a red team engagement before an assumed breach:

1. **Public registration data search**: Search for publicly available registration information to collect system names, IP addresses, operating systems, and technical points of contact.  
   *Source: NIST SP 800-115 Information Security Testing*

2. **DNS server information gathering**: Collect DNS server information to identify network infrastructure and potential entry points.  
   *Source: NIST SP 800-115 Information Security Testing*

3. **Newsgroup postings analysis**: Analyze publicly available newsgroup postings for technical information about the target.  
   *Source: NIST SP 800-115 Information Security Testing*

4. **Public scan databases**: Search online services that publish scan results 

## RAGAS Evaluation

In [24]:
#Configure RAGAS with local Ollama
ragas_client = AsyncOpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

#set up RAGAS with qwen3:4b and nomic embedding.
ragas_llm = llm_factory(
    "qwen3:4b",
    client=ragas_client,
    max_tokens=8192,
    extra_body={"options": {"think": False, "num_ctx": 16384}},
)
ragas_embeddings = embedding_factory(
    "openai", model="nomic-embed-text", client=ragas_client
)

#define RAGAS evaluation metrics
faithfulness_scorer = Faithfulness(llm=ragas_llm)
context_utilization_scorer = ContextUtilization(llm=ragas_llm)
answer_relevancy_scorer = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

print("RAGAS scorers initialised")

RAGAS scorers initialised


In [19]:
#Define 10 user queries
QUERIES = [
    "What reconnaissance techniques should I use before an assumed breach engagement?",
    "How do I establish persistence on a Windows host without triggering EDR?",
    "What are the rules of engagement considerations for a purple team exercise?",
    "How should lateral movement be documented during a red team engagement?",
    "What MITRE ATT&CK techniques are most relevant to Active Directory attacks?",
    "How do I scope a red team engagement for a mid-sized financial organisation?",
    "What is the difference between a red team and a penetration test?",
    "How should I handle discovering a critical zero-day during a client engagement?",
    "What tools are commonly used for command and control in adversary simulation?",
    "How do I write an executive summary for a red team report?",
]

#Pass user queries into RAG pipeline.
results = []
for query in tqdm(QUERIES):
    result = rag_pipeline(query)
    results.append(result)

print(f"\nAll {len(results)} queries complete")

  0%|          | 0/10 [00:00<?, ?it/s]

Query: What reconnaissance techniques should I use before an assumed breach engagement?

Retrieved 10 candidates
Reranked to top 3:
  1. [0.022] NIST SP 800-115 Information Security Testing | 
  2. [0.014] MITRE ATT&CK | T1597.002
  3. [0.014] MITRE ATT&CK | T1596.005
Based on the provided context, here are the appropriate reconnaissance techniques for a red team practitioner to use before a legitimate breach engagement:

1. **Search public registration data** - This includes WHOIS information and other publicly available registration records that can help identify system names, IP addresses, operating systems, and technical points of contact.  
   *Source: NIST SP 800-115 Information Security Testing*

2. **Gather Domain Name System (DNS) server information** - Collecting DNS records helps identify network infrastructure and potential entry points.  
   *Source: NIST SP 800-115 Information Security Testing*

3. **Analyze newsgroup postings and other publicly available information** - 

In [25]:
# Fix the max context and answer characters to mitigate max_token limit issue
MAX_CTX_CHARS = 1200
MAX_ANSWER_CHARS = 500

#Compute RAGAS metrics for all results
async def evaluate_single(query, answer, contexts):
    # Truncate each context chunk before scoring
    contexts = [c[:MAX_CTX_CHARS] for c in contexts]
    answer = answer[:MAX_ANSWER_CHARS]

    #compute RAGAS metrics at the same time.
    faith, ctx_util, relev = await asyncio.gather(
        faithfulness_scorer.ascore(
        user_input=query, response=answer, retrieved_contexts=contexts
    ),
    context_utilization_scorer.ascore(
        user_input=query, response=answer, retrieved_contexts=contexts
    ),
    answer_relevancy_scorer.ascore(
        user_input=query, response=answer),
    )
    
    return {
        "query": query,
        "faithfulness": round(faith.value, 3),
        "context_utilization": round(ctx_util.value, 3),
        "answer_relevancy": round(relev.value, 3),
    }


async def evaluate_all(results):
    records = []
    for r in tqdm(results):
        contexts = [c["text"] for c in r["final_chunks"]]
        scores = await evaluate_single(r["query"], r["answer"], contexts)
        records.append(scores)
        print(
            f"F={scores['faithfulness']:.3f} | "
            f"CU={scores['context_utilization']:.3f} | "
            f"AR={scores['answer_relevancy']:.3f}"
        )
    return records

eval_records = await evaluate_all(results)

  0%|          | 0/10 [00:00<?, ?it/s]

F=1.000 | CU=0.833 | AR=0.877
F=0.500 | CU=1.000 | AR=0.983
F=0.600 | CU=1.000 | AR=0.963
F=0.500 | CU=1.000 | AR=1.000
F=1.000 | CU=0.000 | AR=0.976
F=0.333 | CU=0.333 | AR=0.945
F=0.750 | CU=1.000 | AR=1.000
F=0.500 | CU=0.833 | AR=0.990
F=1.000 | CU=0.583 | AR=0.991
F=0.667 | CU=1.000 | AR=0.888


In [26]:
#Output eval metrics table.
METRIC_COLS = ["faithfulness", "context_utilization", "answer_relevancy"]

df = pd.DataFrame(eval_records)
df.index = [f"Q{i+1}" for i in range(len(df))]

print("RAGAS Evaluation Results")
print(df[["query"] + METRIC_COLS].to_string())

print("\nMean Scores")
print(df[METRIC_COLS].mean().round(3).to_string())

RAGAS Evaluation Results
                                                                                query  faithfulness  context_utilization  answer_relevancy
Q1   What reconnaissance techniques should I use before an assumed breach engagement?         1.000                0.833             0.877
Q2           How do I establish persistence on a Windows host without triggering EDR?         0.500                1.000             0.983
Q3        What are the rules of engagement considerations for a purple team exercise?         0.600                1.000             0.963
Q4            How should lateral movement be documented during a red team engagement?         0.500                1.000             1.000
Q5        What MITRE ATT&CK techniques are most relevant to Active Directory attacks?         1.000                0.000             0.976
Q6       How do I scope a red team engagement for a mid-sized financial organisation?         0.333                0.333             0.945
Q7